# Environmental Data Sources Survey

Before building `dnn_env_terrain` / `pinn_env_terrain`, this notebook does a **quick,
concrete check** on every candidate terrain/wind data source listed in
`documentation/plans_md/Dissertation Plan v5 - 7th June.md` ("Key spatial data resolutions"
and "Static Spatial Data" tables): can we actually reach it, at what real resolution over
Aberfoyle, and what kind of error does it carry?

This is **Tier 1, step 2** of the dissertation plan ("Research and get access to terrain (DEM)
and wind (WASP/Global Wind Atlas) data sources") — see `documentation/handover_2026-07-18.md`
"Next steps". It does not extract per-plot features yet (that's step 3, once a source is
confirmed here); it decides *which* sources are worth building that extraction for.

**How each source is judged, consistently:**

1. **Can we reach it right now?** A live API/download test, not just reading documentation.
2. **What's the real resolution over Aberfoyle specifically** (not the marketing number) —
   and how does that compare to the ~20-40m LiDAR plot grid cell? If several plots share one
   raster cell, any growth difference *between* those plots literally cannot be explained by
   that source, no matter how good the model is — that's a resolution ceiling, not a modelling
   failure.
3. **Measured or modelled?** A modelled/climatological product (like a wind atlas) carries a
   second layer of uncertainty on top of resolution — its own model error — which is easy to
   misattribute to "unexplained biological variance" if not kept separate.
4. **Verdict** — include now, derive later, or defer/exclude, and why.


## Setup

Aberfoyle study-area bounding box, taken directly from the Dissertation Plan's "Study Area
Statistics" table (`EPSG:27700`, from the LiDAR data extent).


In [1]:
import os
import zipfile
from pathlib import Path

# Purpose: Make this notebook work no matter which folder it is opened from.
# Key logic: Walk upward from the current folder until a folder containing
# both README.md and data/ is found -- that is the project root. Same
# pattern as every other notebook in this repo.
notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
print("project_root:", project_root)

import numpy as np
import rasterio
import requests
from pyproj import Transformer
from rasterio.windows import from_bounds

# Aberfoyle bounding box, EPSG:27700 (British National Grid) -- see the Dissertation Plan's
# "Study Area Statistics" table.
BBOX_27700 = dict(minx=235_000, miny=692_500, maxx=255_000, maxy=707_500)

# Every downloaded file is cached here (gitignored -- same convention as data/raw/ already
# used for the main LiDAR GeoPackage) so re-running this notebook doesn't re-download.
CACHE_DIR = project_root / "data" / "raw" / "environmental"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Cache directory:", CACHE_DIR.resolve())


project_root: /Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss


Cache directory: /Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/data/raw/environmental


## 1. OS Terrain 50 DTM

**What it gives us:** elevation, from which slope, northness/eastness (aspect), TWI
(topographic wetness index), and TOPEX (wind shelter) can all be *derived* -- so this one
source underpins four to five of the planned terrain features. Free OS OpenData.

**Access test:** the OS Downloads API needs no API key for this product's metadata or the
data itself -- both work as plain unauthenticated HTTP requests.


In [2]:
# 1a. Confirm the product is listed and reachable, no API key needed.
resp = requests.get("https://api.os.uk/downloads/v1/products/Terrain50", timeout=15)
resp.raise_for_status()
product = resp.json()
print("Product:", product["name"])
print("Formats available:", [f["format"] for f in product["formats"]])

# 1b. List the actual downloadable files -- only whole-Great-Britain bundles are offered
# (no more per-tile downloads via this API), so the smallest format is the practical choice.
resp = requests.get(f"{product['downloadsUrl']}", timeout=15)
resp.raise_for_status()
downloads = resp.json()
for item in downloads:
    print(f"{item['format']:35s} {item['size'] / 1e6:8.1f} MB   {item['fileName']}")


Product: OS Terrain® 50
Formats available: ['ASCII Grid and GML (Grid)', 'ESRI® Shapefile', 'GeoPackage', 'GML', 'Vector Tiles']


ASCII Grid and GML (Grid)              161.7 MB   terr50_gagg_gb.zip
ESRI® Shapefile                       1045.9 MB   terr50_cesh_gb.zip
GML                                   1124.3 MB   terr50_cgml_gb.zip
GeoPackage                            1233.5 MB   terr50_gpkg_gb.zip
Vector Tiles                          1070.8 MB   terr50_mbtiles_gb.zip


In [3]:
# 1c. Download the smallest bundle (ASCII Grid + GML, ~160MB, one-time cost, cached below),
# then pull out just the one 10km tile that covers the Aberfoyle grid centre (NN40 -- matches
# xllcorner=240000, yllcorner=700000, right in the middle of the study bbox above). This is a
# spot-check, not full extraction: the real Tier-1 extraction step will need every tile that
# intersects BBOX_27700 (NN and NS tiles both, since the bbox straddles that 100km-square
# boundary), not just this one.
gb_zip_path = CACHE_DIR / "terr50_gagg_gb.zip"
ascii_grid = next(f for f in downloads if f["format"] == "ASCII Grid and GML (Grid)")

if not gb_zip_path.exists():
    resp = requests.get(ascii_grid["url"], timeout=120)
    resp.raise_for_status()
    gb_zip_path.write_bytes(resp.content)
print(f"GB bundle cached at {gb_zip_path} ({gb_zip_path.stat().st_size / 1e6:.1f} MB)")

with zipfile.ZipFile(gb_zip_path) as gb_zip:
    tile_zip_bytes = gb_zip.read("data/nn/nn40_OST50GRID_20260529.zip")

tile_zip_path = CACHE_DIR / "nn40_tile.zip"
tile_zip_path.write_bytes(tile_zip_bytes)
with zipfile.ZipFile(tile_zip_path) as tile_zip:
    tile_zip.extractall(CACHE_DIR)
print("Extracted NN40.asc")


GB bundle cached at /Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/data/raw/environmental/terr50_gagg_gb.zip (161.7 MB)
Extracted NN40.asc


In [4]:
# 1d. Read the ASCII grid directly (it's a plain text DTM, no GIS library needed) and confirm
# real elevation values inside Aberfoyle, plus the actual resolution (should be 50m).
asc_path = CACHE_DIR / "NN40.asc"
with open(asc_path) as f:
    header = {}
    for _ in range(5):
        key, value = f.readline().split()
        header[key.lower()] = float(value)
    elevation = np.loadtxt(f)

print("Header:", header)
print("Grid shape:", elevation.shape)
print("Resolution (cellsize):", header["cellsize"], "m")
print(f"Elevation range over this tile: {elevation.min():.1f}m to {elevation.max():.1f}m "
      f"(mean {elevation.mean():.1f}m)")


Header: {'ncols': 200.0, 'nrows': 200.0, 'xllcorner': 240000.0, 'yllcorner': 700000.0, 'cellsize': 50.0}
Grid shape: (200, 200)
Resolution (cellsize): 50.0 m
Elevation range over this tile: 21.6m to 726.1m (mean 248.6m)


**Resolution over Aberfoyle:** confirmed 50m post spacing (`cellsize=50` in the header, not
just the documented spec). One tile alone (`NN40`, 200x200 cells = 10km x 10km) already covers
a meaningful chunk of the study area; full coverage needs the handful of `NN`/`NS` tiles the
bbox actually spans.

**Error / variance:** two kinds, and they should not be conflated:

- **Resolution ceiling, not error**: LiDAR plots are ~20-40m grid cells; a 50m DTM cell can
  contain 2-6 of them. Any elevation/slope/TWI-derived feature is therefore *identical* for
  every plot inside the same DTM cell -- so if two such plots have genuinely different growth,
  DTM-derived terrain cannot be the explanation for that specific difference. This caps the
  finest spatial resolution any terrain-only attribution claim can make, independent of model
  quality.
- **Genuine measurement error**: OS Terrain 50 is itself interpolated from contour/spot-height
  data, not a raw DSM/DTM survey at 50m native resolution -- its own technical specification
  states vertical accuracy varies with terrain steepness (worse on steep slopes, which
  Aberfoyle has plenty of). This is real data-source error, separate from the resolution
  ceiling above.

**Verdict: include.** Free, no key, confirmed live, correct area and resolution. Elevation,
slope, northness, eastness, TWI, and TOPEX (see below) all derive from this one source.


## 2. TOPEX (wind shelter index) -- derived, not a separate source

**What it gives us:** a wind-exposure/shelter proxy computed directly from the DTM above (sum
of horizon angles to the surrounding terrain in several compass directions, at a chosen search
radius) -- not something to download separately, so there's no API to test here.

**Resolution:** inherits OS Terrain 50's 50m grid -- same ceiling and same caveats as above.

**Error / variance:** the main source of "error" here is a *methodological choice*, not a data
quality issue -- the search radius and number of compass directions used are modelling
decisions (Worrell 1987, cited in the plan, is the reference point for a defensible radius).
Once implemented, this should get a short sensitivity check (does the shelter ranking of plots
change much across a couple of reasonable radius choices?) rather than being treated as a fixed
ground truth the way a directly-measured source would be.

**Verdict: include**, derived during the Tier-1 terrain extraction step once OS Terrain 50 is
pulled for the full study area (not repeated per-source here since it needs no separate
access/download check).


## 3. WASP wind atlas

**What it gives us:** project-specific mean wind speed / exposure at (or near) canopy height --
the plan's preferred wind source, since it's purpose-built rather than a generic public
climatology.

**Access test: not applicable.** This is not a public API or download -- access is via Dr
Suárez-Minguez (project contact), and per the handover doc this contact/data request has not
been made yet. Nothing to test until that happens.

**Resolution:** stated as ~200m-1km in the plan's own table, but genuinely "confirm" --
unverified until the actual data is in hand.

**Error / variance:** unknown until received -- could be a measured network of stations
(different error profile: sparse but locally accurate) or itself a modelled wind atlas at
higher resolution than Global Wind Atlas (different error profile: denser but still
model-uncertain). Worth asking about this distinction specifically when requesting the data,
since it changes how its error should be described in the write-up.

**Verdict: defer.** Action item: email Dr Suárez-Minguez to request the data (per the plan's
Day 1-2 timeline, this is already slightly behind schedule). Use Global Wind Atlas (below) as
the working wind source in the meantime -- exactly the fallback the plan itself specifies.


## 4. Global Wind Atlas

**What it gives us:** a public, modelled wind-speed climatology, available at several heights
above ground (10/50/100/150/200m) -- the plan's explicit WASP fallback.

**Access test:** a public REST API, no key needed, but only served as a whole-country GeoTIFF
per height (not a windowed/point query for this bulk endpoint), so a full-country file has to
be downloaded once and then read locally.


In [5]:
# 4a. Download the whole-GB wind-speed layer at 10m height -- the plan's guidance is to use
# the lowest suitable height above ground rather than turbine-hub-height layers.
gwa_path = CACHE_DIR / "gbr_wind_10m.tif"
if not gwa_path.exists():
    resp = requests.get(
        "https://globalwindatlas.info/api/gis/country/GBR/wind-speed/10", timeout=60
    )
    resp.raise_for_status()
    gwa_path.write_bytes(resp.content)
print(f"Global Wind Atlas GBR/10m cached at {gwa_path} ({gwa_path.stat().st_size / 1e6:.1f} MB)")


Global Wind Atlas GBR/10m cached at /Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/data/raw/environmental/gbr_wind_10m.tif (63.7 MB)


In [6]:
# 4b. Windowed read over the Aberfoyle bbox only (reprojecting the bbox from EPSG:27700 into
# the raster's own CRS), and check both the wind-speed values and the *real* ground resolution
# at this latitude -- GWA's headline "250m" is a longitude/latitude grid spacing, which is NOT
# 250m on the ground once you're away from the equator.
with rasterio.open(gwa_path) as ds:
    print("Raster CRS:", ds.crs)
    print("Pixel size (degrees):", ds.res)

    transformer = Transformer.from_crs("EPSG:27700", ds.crs, always_xy=True)
    minx, miny = transformer.transform(BBOX_27700["minx"], BBOX_27700["miny"])
    maxx, maxy = transformer.transform(BBOX_27700["maxx"], BBOX_27700["maxy"])

    window = from_bounds(minx, miny, maxx, maxy, ds.transform)
    wind_speed = ds.read(1, window=window)

    lat_mid = (miny + maxy) / 2
    import math
    metres_per_degree_lon = 111_320 * math.cos(math.radians(lat_mid))
    metres_per_degree_lat = 111_320
    ew_resolution_m = ds.res[0] * metres_per_degree_lon
    ns_resolution_m = ds.res[1] * metres_per_degree_lat

print("Windowed grid shape over Aberfoyle:", wind_speed.shape)
print(f"Wind speed (m/s) at 10m: min={wind_speed.min():.2f}, max={wind_speed.max():.2f}, "
      f"mean={wind_speed.mean():.2f}")
print(f"Real ground resolution at Aberfoyle's latitude (~{lat_mid:.1f} deg N): "
      f"{ew_resolution_m:.0f}m (east-west) x {ns_resolution_m:.0f}m (north-south)")


Raster CRS: EPSG:4326
Pixel size (degrees): (0.002500000000000001, 0.002500000000000001)


Windowed grid shape over Aberfoyle: (56, 125)
Wind speed (m/s) at 10m: min=0.07, max=18.77, mean=3.60
Real ground resolution at Aberfoyle's latitude (~56.2 deg N): 155m (east-west) x 278m (north-south)


**Resolution over Aberfoyle:** the advertised "250m" is a *0.0025-degree* grid, which is only
250m north-south -- at Aberfoyle's latitude (~56N) it is genuinely narrower east-west (see the
printed figure above, expect roughly 140-155m) because a degree of longitude covers less
ground at higher latitudes. Worth stating precisely rather than quoting "250m" flat, since it's
finer than the DTM in one direction and coarser in the other.

**Error / variance:** this is a *modelled* long-term climatology (a mesoscale wind model
output), not a direct measurement -- it will have systematic uncertainty that grows in
topographically complex terrain (Aberfoyle's steep glens are exactly this kind of terrain), on
top of its native grid resolution. Any "wind exposure explains growth anomaly X" claim built on
GWA is bounded by *both* of these -- the resolution ceiling (as with the DTM above) and this
extra model-uncertainty layer, which a directly-measured source like WASP would not carry to
the same degree.

**Verdict: include now.** Confirmed live and reachable, real values inside the study area, and
already the plan's designated fallback -- gives a working wind feature while WASP access is
pending.


## 5. Other sources considered, not tested live here

The plan's "Key spatial data resolutions" table already gives the reasoning for these; not
repeated with a live API test since the resolution numbers alone rule them out (or make them
conditional) for *plot-level spatial* attribution, which is what this stage needs:

- **HadUK-Grid** (rainfall/temperature, 1km) -- ~300 cells over the whole 300km2 forest, so a
  single value is shared by many plots. Genuinely useful later for *interval-level* (temporal)
  climate indices (SQ2), not spatial attribution -- deferred to that later stage, not excluded
  outright.
- **ERA5-Land** (9km) -- the entire forest falls inside ~3-4 cells. No within-forest spatial
  signal possible at all. Excluded.
- **CEH / UK soil products** -- resolution is product-dependent and often coarse/polygonal. Per
  the plan's own rule, only worth extracting if a specific product shows meaningful
  within-forest variation after extraction -- not assumed either way here, just not tested this
  pass since terrain/wind are the priority per the handover's next-steps list.
- **James Hutton 1:250,000 soil map** -- ~3-5 polygons cover the *entire* study area. No
  within-forest variation possible. Excluded.


## Summary and recommendation

| Source | Reachable now? | Real resolution over Aberfoyle | Measured or modelled | Verdict |
|---|---|---|---|---|
| OS Terrain 50 DTM | Yes, no key | 50m confirmed | Measured (interpolated from survey) | **Include** -- elevation, slope, northness, eastness, TWI |
| TOPEX | N/A (derived) | 50m (inherits DTM) | Derived, radius is a modelling choice | **Include**, derive alongside DTM |
| WASP | No -- contact pending | ~200m-1km, unconfirmed | Unknown until received | **Defer** -- email Dr Suarez-Minguez |
| Global Wind Atlas | Yes, no key | ~140-155m (E-W) x 250m (N-S), confirmed | Modelled climatology | **Include now** as WASP fallback |
| HadUK-Grid | Not tested | 1km | Measured/interpolated | Later -- temporal indices only |
| ERA5-Land | Not tested | 9km | Modelled reanalysis | Excluded -- too coarse |
| CEH soil | Not tested | Product-dependent | Mixed | Untested -- only if shown to vary |
| James Hutton soil | Not tested | 1:250,000 | Mapped | Excluded -- too coarse |

**What this means for `dnn_env_terrain` / `pinn_env_terrain`:** start with the two confirmed,
spatially-varying sources -- OS Terrain 50 (elevation, slope, northness, eastness, TWI, TOPEX)
as the **terrain** group, and Global Wind Atlas as a **wind** group of one feature -- then add
WASP as a second wind feature (or a replacement, if it turns out more reliable) once access
comes through. This matches the plan's own Env-PINN v2 (terrain-only) -> v3 (terrain + wind)
staging, and the "build up slowly" approach: nothing here commits to a final feature set, it
just confirms what's real and usable to start with.
